In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Tue Jan 27 20:30:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd

In [19]:
df = pd.read_excel("/content/protein_constructs_w_label_masks.xlsx")[['rcsb_id', 'rcsb_entity_ids', 'uniprot_seq','pbd_id', 'pdb_sequence_sanitized', 'label_mask']]
df.head()

,rcsb_id,rcsb_entity_ids,uniprot_seq,pbd_id,pdb_sequence_sanitized,label_mask
0,2GD8,1,MSHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKP...,SHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKPL...,SHHWGYGKHNGPEHWHKDFPIAKGERQSPVDIDTHTAKYDPSLKPL...,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,2GDD,1,MLNLLLLALPVLASRAYAAPAPGQALQRVGIVGGQEAPRSKWPWQV...,IVGGQEAPRSKWPWQVSLRVHGPYWMHFCGGSLIHPQWVLTAAHCV...,IVGGQEAPRSKWPWQVSLRVHGPYWMHFCGGSLIHPQWVLTAAHCV...,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,2GDE,1,MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRANT...,TFGSGEADCGLRPLFEKKSLEDKTERELLESYIDGR,TFGSGEADCGLRPLFEKKSLEDKTERELLESYIDGR,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,2GDE,2,MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRANT...,IVEGSDAEIGMSPWQVMLFRKSPQELLCGASLISDRWVLTAAHCLL...,IVEGSDAEIGMSPWQVMLFRKSPQELLCGASLISDRWVLTAAHCLL...,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,2GDO,1,MAVPFVEDWDLVQTLGEGAYGEVQLAVNRVTEEAVAVKIVDMKRAV...,MAVPFVEDWDLVQTLGEGAYGEVQLAVNRVTEEAVAVKIVDMKRAV...,MAVPFVEDWDLVQTLGEGAYGEVQLAVNRVTEEAVAVKIVDMKRAV...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [20]:
# model_name = "facebook/esm2_t33_650M_UR50D"

# tokenizer = AutoTokenizer.from_pretrained(model_name, do_lower_case=False)
# model = AutoModel.from_pretrained(model_name)

# model.eval()

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [22]:
from tqdm.auto import tqdm

In [23]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import T5EncoderModel, AutoTokenizer
hf_token = "***REMOVED***"
model_name = "ElnaggarLab/ankh-large"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    do_lower_case=False,
    token=hf_token
)

model = T5EncoderModel.from_pretrained(
    model_name,
    token=hf_token
)
model.to(device).eval()

Loading ElnaggarLab/ankh-large...


T5EncoderModel(
  (shared): Embedding(144, 1536)
  (encoder): T5Stack(
    (embed_tokens): Embedding(144, 1536)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1536, out_features=1024, bias=False)
              (k): Linear(in_features=1536, out_features=1024, bias=False)
              (v): Linear(in_features=1536, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1536, bias=False)
              (relative_attention_bias): Embedding(64, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1536, out_features=3840, bias=False)
              (wi_1): Linear(in_features=1536, out_features=3840, bias=False)
              (wo): Lin

In [24]:
df['seq_len'] = df['uniprot_seq'].str.len()
df = df.sort_values('seq_len').reset_index(drop=True)

In [31]:
inputs = tokenizer(
    df['uniprot_seq'].tolist()[0],
    truncation=True,
    max_length=2048,
    padding=False,
    return_tensors="pt"
)

inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

embeddings = outputs.last_hidden_state

print(f"Full embedding shape: {embeddings.shape}")

protein_vector = torch.mean(embeddings[0, :-1, :], dim=0)
print(f"Mean pooled vector shape: {protein_vector.shape}")
protein_vector

Full embedding shape: torch.Size([1, 16, 1536])
Mean pooled vector shape: torch.Size([1536])


tensor([ 0.0195,  0.0034, -0.0078,  ..., -0.0148, -0.0095,  0.0044],
       device='cuda:0')

In [35]:


print("Tokenizing sequences...")
encodings = tokenizer(
    df['uniprot_seq'].tolist(),
    truncation=True,
    max_length=2048,
    padding=False
)

class FastSeqDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        return {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}

dataset = FastSeqDataset(encodings)
batch_size = 16

def collate_fn(batch):
    return tokenizer.pad(batch, padding=True, return_tensors="pt")

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

all_embeddings = []

print("Starting embedding generation...")
model.eval()

with torch.no_grad():
    with torch.cuda.amp.autocast(enabled=False):
        for batch in tqdm(dataloader, desc="Ankh Generation"):
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            last_hidden = outputs.last_hidden_state.float() # Ensure we are in FP32

            mask = batch['attention_mask'].unsqueeze(-1).expand(last_hidden.size()).float()
            sum_embeddings = torch.sum(last_hidden * mask, dim=1)
            actual_counts = torch.clamp(mask.sum(1), min=1e-9)
            seq_embeddings = sum_embeddings / actual_counts

            all_embeddings.append(seq_embeddings.cpu())

final_embeddings = torch.cat(all_embeddings, dim=0)
torch.save(final_embeddings, "ankh_embeddings.pt")
print(f"Success! Final shape: {final_embeddings.shape}")

Tokenizing sequences...
Starting embedding generation...


/tmp/ipython-input-2714809797.py:72: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Ankh Generation:   0%|          | 0/4421 [00:00<?, ?it/s]

You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` m

Success! Final shape: torch.Size([70732, 1536])


In [36]:
final_embeddings

tensor([[ 0.0182,  0.0036, -0.0069,  ..., -0.0138, -0.0081,  0.0042],
        [ 0.0182,  0.0036, -0.0069,  ..., -0.0138, -0.0081,  0.0042],
        [ 0.0038,  0.0039, -0.0080,  ..., -0.0119, -0.0006, -0.0042],
        ...,
        [ 0.0234, -0.0014,  0.0015,  ...,  0.0030,  0.0049,  0.0034],
        [ 0.0234, -0.0014,  0.0015,  ...,  0.0030,  0.0049,  0.0034],
        [ 0.0234, -0.0014,  0.0015,  ...,  0.0030,  0.0049,  0.0034]])

In [37]:
df['embeddings'] = list(final_embeddings.numpy())

In [38]:
df.to_parquet("protein_data.parquet")

In [39]:
df.to_pickle("protein_data.pickle")